# 00 — LOCK 확인

**무엇을 확인하는가**

1. submodule 이 초기화되어 있는가 (가장 흔한 함정)
2. `config.env` 가 있는가
3. LOCK.md 의 항목이 지금 상태와 맞는가
4. 코드 앵커가 아직 유효한가
5. findings 레코드가 기록 요건을 지키는가

**여기서 걸리면 아래 노트북은 볼 필요가 없습니다.** 무엇을 보고 있는지
모르는 채로 숫자를 내게 됩니다.

## 0. 부트스트랩

In [ ]:
import sys
from pathlib import Path

# 노트북에서 저장소 루트를 import 경로에 넣습니다.
REPO = Path.cwd()
while not (REPO / "run.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("저장소 루트:", REPO)

## 1. submodule — 가장 흔한 함정

`--recursive` 없이 clone 하면 `upstream/BatteryMFormer/` **만** 빈 폴더가
됩니다. 형제 폴더는 차 있어서 알아채기 어렵습니다.

In [ ]:
for name in ("BatteryML", "BatteryLife", "BatteryMFormer"):
    path = REPO / "upstream" / name
    files = list(path.rglob("*")) if path.exists() else []
    count = sum(1 for p in files if p.is_file())
    print(f"{name:16} {count:5}개 파일  {path}")

mformer = REPO / "upstream" / "BatteryMFormer"
if not mformer.exists() or not any(mformer.iterdir()):
    print()
    print("!" * 70)
    print("BatteryMFormer 가 비어 있습니다. submodule 이 초기화되지 않았습니다.")
    print()
    print("    git submodule update --init")
    print()
    print("형제 폴더는 차 있어서 헷갈리기 쉬운 지점입니다.")
    print("!" * 70)
else:
    print("\nsubmodule ok")

## 2. config.env

없으면 `config.env.example` 이 대신 읽힙니다. 그 경우 경로가 예시값이라
데이터를 못 찾습니다.

In [ ]:
from verify import load_config

config_path = REPO / "config.env"
print("config.env:", "있음" if config_path.exists() else "없음 (example 을 대신 읽습니다)")
if not config_path.exists():
    print("\n    copy config.env.example config.env\n")

config = load_config()
for key in ("DATA_ROOT", "ZENODO_DIR", "EXTRACT_DIR", "HF_DIR", "CKPT_ROOT",
            "CUDA_DEVICES", "NUM_PROCESSES", "NUM_WORKERS", "BATCH_SIZE"):
    value = config.get(key, "(없음)")
    exists = ""
    if key.endswith(("_DIR", "_ROOT")) and value not in ("", "(없음)"):
        exists = "  [있음]" if Path(value).exists() else "  [경로 없음]"
    print(f"  {key:14} {value}{exists}")

## 3. LOCK 대조

`(미정)` 이 남아 있는 것은 정상입니다 — 데이터를 받고 01·03 을 돌린 뒤
`python run.py lock-init` 으로 채웁니다.

**불일치** 가 나오면 어느 층인지 보십시오. 코드 · 데이터 · 환경이 다 맞는데
결과만 다르면 비결정성 문제이고, 그 자체가 발견입니다.

In [ ]:
from verify import lock

result = lock.check()
print(lock.format_report(result))

## 4. 코드 앵커

행 번호는 상위 갱신 시 썩습니다. `행이동` 은 문제가 아니고 `anchors.yaml`
의 `line` 만 고치면 됩니다. **`내용변경` · `소실` 은 그 앵커에 걸린 findings
레코드를 다시 봐야 한다는 뜻입니다.**

In [ ]:
from verify import anchors

rows = anchors.check()
print(anchors.format_report(rows))

## 5. findings 기록 요건

`확인` 인데 `locus` 가 없거나, `부재확인` 인데 `searched` 가 없으면 위반입니다.
`--dry-run` 이므로 문서를 다시 쓰지는 않습니다.

In [ ]:
from verify import render

summary = render.render_all(write=False)
print(f"레코드 {summary['records']}개")
for problem in summary["problems"]:
    print("  -", problem)
if not summary["problems"]:
    print("기록 요건 위반 없음")

## 6. 판정 분포

지금 시점의 판정입니다. **`미정` 이 많은 것이 정상입니다** — 논문 슬롯이
아직 비어 있기 때문입니다. `미정` 을 줄이려고 슬롯을 억지로 채우지
마십시오. 안 찾아본 것은 안 찾아본 것입니다.

In [ ]:
from verify import load_yaml, FINDINGS
from verify.render import derive_verdict

document = load_yaml(FINDINGS / "registry.yaml")
counts = {}
for record in document["records"]:
    verdict, _ = derive_verdict(record)
    counts[verdict] = counts.get(verdict, 0) + 1

for verdict, count in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"  {verdict:20} {count}")

print()
print("확인불가는 미해결이 아닙니다. 종결된 판정입니다.")
print("docs/OPEN_QUESTIONS.md 의 별도 절을 보십시오.")